# 02 — Exploratory Data Analysis and Visualisation
Label distribution, common unigram/bigram terms, and word cloud, adapting the report's visualisation section.

In [ ]:
!pip -q install nltk wordcloud seaborn scikit-learn
import re, numpy as np, pandas as pd, nltk, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
for r in ["punkt","punkt_tab","stopwords","wordnet","omw-1.4"]: nltk.download(r, quiet=True)

def clean_tweet(tweet):
    if pd.isna(tweet): return ""
    tweet=str(tweet)
    tweet=re.sub(r"https?://\S+|www\.\S+", " url ", tweet, flags=re.I)
    tweet=re.sub(r"@\w+", " user ", tweet)
    tweet=re.sub(r"#(\w+)", r"\1", tweet)
    tweet=re.sub(r"[^\w\s]", " ", tweet)
    return re.sub(r"\s+", " ", tweet).strip().lower()

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True, custom_stopwords=None):
    tokens=word_tokenize(clean_tweet(text))
    stops=set(stopwords.words("english")) if remove_stopwords else set()
    stops.update(custom_stopwords or [])
    tokens=[w for w in tokens if w not in stops and w.strip()]
    if stem:
        stemmer=PorterStemmer(); tokens=[stemmer.stem(w) for w in tokens]
    if lemmatize:
        lemma=WordNetLemmatizer(); tokens=[lemma.lemmatize(w) for w in tokens]
    return " ".join(tokens)

def load_csv():
    uploaded=files.upload()
    name=next((n for n in uploaded if n.lower().endswith(".csv")),None)
    if not name: raise ValueError("Upload a CSV file.")
    data=pd.read_csv(name)
    if not {"tweet_text","label"}.issubset(data.columns):
        raise ValueError("CSV must contain columns named tweet_text and label. Rename your columns if needed.")
    data=data.dropna(subset=["label"]).copy()
    data["tweet_text"]=data["tweet_text"].fillna("").astype(str)
    return data

In [ ]:
df=load_csv()
df["tweet_text_clean"]=df["tweet_text"].apply(clean_tweet)
display(df["label"].value_counts().rename_axis("label").to_frame("count"))
df["label"].value_counts().plot(kind="bar",figsize=(9,4),title="Records per Label")
plt.xlabel("Label"); plt.ylabel("Count"); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

## Top unigram and bigram terms by label

In [ ]:
for label in df["label"].astype(str).unique():
    text=" ".join(df.loc[df["label"].astype(str)==label,"tweet_text_clean"])
    if not text.strip(): continue
    for name,ngram in [("Unigram",(1,1)),("Bigram",(2,2))]:
        v=CountVectorizer(stop_words=["user","url","rt","mkr"],ngram_range=ngram,max_features=10)
        mat=v.fit_transform([text]); counts=np.asarray(mat.sum(axis=0)).ravel(); terms=v.get_feature_names_out()
        order=np.argsort(counts)[::-1]
        plt.figure(figsize=(9,4)); plt.barh(terms[order][::-1],counts[order][::-1])
        plt.title(f"Top {name} Terms — {label}"); plt.xlabel("Count"); plt.tight_layout(); plt.show()

## Word cloud

In [ ]:
from wordcloud import WordCloud, STOPWORDS
text=" ".join(df["tweet_text_clean"])
if text.strip():
    cloud=WordCloud(width=1400,height=700,background_color="white",
      stopwords=set(STOPWORDS).union({"url","user","rt","mkr"}),collocations=False).generate(text)
    plt.figure(figsize=(14,7)); plt.imshow(cloud,interpolation="bilinear"); plt.axis("off"); plt.show()
else: print("No text available.")